In [33]:
import pandas as pd
from datasets import load_dataset

ds = load_dataset("databricks/databricks-dolly-15k", split="train")
df = ds.to_pandas()

In [34]:
#Returns the instructions + context(if applicable) from each row
def build_query(row):
    if row["context"]:
        return f"{row['instruction']}\n\nContext:\n{row['context']}"
    return row['instruction']

#changes the category name from "category" -> "source category", condenses dataframe to two columns ("query", "source_category")
df["query"] = df.apply(build_query, axis = 1)
df = df.rename(columns={'category' : 'source_category'})
df = df[["query", "source_category"]].copy()

In [35]:
#Remove duplicates form query
df = df.drop_duplicates(subset="query").reset_index(drop=True)

In [36]:
#gets the query word count
df["query_word_count"] = df["query"].str.split().str.len()

In [47]:
#Samples 3000 queries over 8 categories
df_sampled = df.sample(n = min(3000, len(df)), random_state=42).reset_index(drop=True)

#prints out source category sample counts and descriptive information regarding word counts
print(f"Sampled {len(df_sampled)} queries across {df_sampled['source_category'].nunique()} categories")
print(df_sampled["source_category"].value_counts())
print(df_sampled["query_word_count"].describe())

Sampled 3000 queries across 8 categories
source_category
open_qa                   735
general_qa                440
classification            426
closed_qa                 375
brainstorming             345
information_extraction    304
summarization             217
creative_writing          158
Name: count, dtype: int64
count    3000.000000
mean       67.097667
std       163.441789
min         1.000000
25%         8.000000
50%        13.000000
75%        77.250000
max      3804.000000
Name: query_word_count, dtype: float64


In [48]:
#Saves queries dataset
df_sampled.to_parquet('/Users/vledwards09/Desktop/personal projects/LLM-Router-Project/data/raw_queries_parquet', index=False)